# Alignment scores against child allele consensuses

Plot a trio `--alignment-scores` TSV as two stacked panels, one per child candidate as seen in the preprint. 

The example uses **`examples/example-denovo-scores.txt`**.

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "font.size": 11,
    "text.usetex": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

## Settings

`TRID` selects exactly one locus. `CANDIDATE_INDICES=None` uses its two candidate indices in ascending order; set a pair such as `[1, 0]` to choose the panel order. The suffixes in $M_i$, $F_i$, and $C_i$ refer to **allele indices**, not VCF genotype values, so distinct alleles with the same genotype value remain separate.

In [ ]:
INPUT_PATH = next(
    (directory / "examples/example-denovo-scores.txt"
     for directory in (Path.cwd(), *Path.cwd().parents)
     if (directory / "examples/example-denovo-scores.txt").is_file()),
    Path("examples/example-denovo-scores.txt"),
)

# For your own data, replace INPUT_PATH with Path("/path/to/scores.tsv") and a chosen TRID.
TRID = "example-denovo-locus"
CANDIDATE_INDICES = None
FIGSIZE = (11, 8.2)
RANDOM_SEED = 42
OUTPUT_STEM = None

## Read the score table

In [ ]:
scores = pd.read_csv(INPUT_PATH, sep="\t", dtype={"trid": str, "read_id": str})
locus_scores = scores.loc[scores["trid"] == TRID]
if locus_scores.empty:
    raise ValueError(f"TRID {TRID!r} not found. Available TRIDs: {scores['trid'].unique().tolist()}")

print(f"Loaded {len(scores):,} score rows from {INPUT_PATH.name}")
locus_scores.groupby(
    ["candidate_index", "candidate_genotype", "sample_role", "source_index", "source_genotype"],
    sort=True,
)["score"].agg(n="size", median="median", minimum="min", maximum="max")

## Two-panel figure

In [ ]:
def plot_alignment_scores(scores, trid, candidate_indices=None, *, figsize=(11, 8.2), seed=42):
    locus = scores.loc[scores["trid"] == trid]
    if locus.empty:
        raise ValueError(f"TRID {trid!r} not found in the score table")

    available = sorted(locus["candidate_index"].unique())
    candidates = available if candidate_indices is None else list(candidate_indices)
    if len(candidates) != 2 or len(set(candidates)) != 2:
        raise ValueError("Choose exactly two distinct candidate indices for the two panels")
    if not set(candidates).issubset(available):
        raise ValueError(f"Candidate indices must be drawn from {available}")

    selected = locus.loc[
        locus["candidate_index"].isin(candidates)
        & (
            locus["sample_role"].isin(["mother", "father"])
            | (
                (locus["sample_role"] == "child")
                & (locus["source_index"] == locus["candidate_index"])
            )
        )
    ]
    if selected.empty:
        raise ValueError("No trio alignment scores for the selected candidates")

    colours = {
        "mother": ("#cf7b32", "#929044"),
        "father": ("#895491", "#913f50"),
        "child": ("#527e99",),
    }
    prefixes = {"mother": "M", "father": "F", "child": "C"}
    low, high = selected["score"].min(), selected["score"].max()
    span = max(float(high - low), 1.0)
    limits = (low - 0.20 * span, high + 0.10 * span)
    rng = np.random.default_rng(seed)
    fig, axes = plt.subplots(2, 1, figsize=figsize, sharex=True, layout="constrained")

    for ax, candidate in zip(axes, candidates):
        panel = selected.loc[selected["candidate_index"] == candidate]
        groups = [
            (role, source)
            for role in ("mother", "father", "child")
            for source in sorted(panel.loc[panel["sample_role"] == role, "source_index"].unique())
        ]
        for position, (role, source) in enumerate(groups):
            reads = panel.loc[
                (panel["sample_role"] == role) & (panel["source_index"] == source)
            ].sort_values("read_id")
            values = reads["score"].to_numpy(dtype=float)
            colour = colours[role][int(source) % len(colours[role])]

            if len(values) > 1 and np.ptp(values) > 0:
                violin = ax.violinplot(
                    values, positions=[position], orientation="horizontal",
                    widths=0.72, showextrema=False, showmedians=True, bw_method="scott",
                )
                body = violin["bodies"][0]
                body.set_facecolor("#d4d5d3")
                body.set_edgecolor("#a9adaa")
                body.set_alpha(0.85)
                body.set_linewidth(0.8)
                violin["cmedians"].set_color(colour)
                violin["cmedians"].set_linewidth(1.4)
            else:
                ax.plot(
                    [values[0], values[0]], [position - 0.25, position + 0.25],
                    color=colour, linewidth=1.4, zorder=2,
                )

            ax.scatter(
                values, position + rng.uniform(-0.25, 0.25, size=len(values)),
                s=13, color=colour, alpha=0.85, linewidths=0, zorder=3,
            )
            anchor = (values.min(), position)
            ax.annotate(
                rf"$\mathbf{{{prefixes[role]}}}_{{{int(source)}}}$", anchor,
                xytext=(-10, -3), textcoords="offset points",
                ha="right", va="center", fontsize=19, color="#2e3033",
            )
            ax.annotate(
                f"n={len(values)}", anchor, xytext=(-10, 15), textcoords="offset points",
                ha="right", va="center", fontsize=10, color="#2e3033",
            )

        ax.set_ylim(len(groups) - 0.4, -0.6)
        ax.set_xlim(limits)
        ax.set_yticks([])
        ax.set_ylabel(rf"$A_i$ to $C_{{{int(candidate)}}}$ consensus", fontsize=15, labelpad=18)
        ax.set_axisbelow(True)
        ax.grid(axis="x", color="#e6e7e5", linestyle="--", linewidth=0.8)
        ax.tick_params(axis="x", labelbottom=True, length=0, pad=8, colors="#414549")
        for spine in ax.spines.values():
            spine.set_color("#d5d8d5")
            spine.set_linewidth(6)

    axes[-1].set_xlabel("Alignment score", fontsize=15, labelpad=10)
    fig.suptitle(trid, fontsize=13, color="#414549")
    return fig, axes

In [ ]:
fig, axes = plot_alignment_scores(
    scores, TRID, CANDIDATE_INDICES, figsize=FIGSIZE, seed=RANDOM_SEED,
)
plt.show()